<a href="https://colab.research.google.com/github/ishwarraja/SOAI/blob/main/ERAv4/S7/S7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# cifar10_dilated_ds.py
"""
CIFAR-10 model with:
- No MaxPool (last conv uses stride=2)
- Depthwise Separable Conv present
- Dilated conv(s) present
- GAP + FC head
- Albumentations: HorizontalFlip, ShiftScaleRotate, CoarseDropout
- Prints effective receptive field (should be > 44)
- Params < 200k (about ~120-140k for this config)
- Training loop (SGD + cosine LR) + validation
"""

import os
import math
import time
import argparse
from collections import deque

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, ToPILImage

import albumentations as A
from albumentations.pytorch import ToTensorV2

# ------------------------
# Utils
# ------------------------
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def device_and_seed(seed=42):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    return device

# ------------------------
# Albumentations wrapper for CIFAR-10
# ------------------------
class AlbumentationsTransform:
    def __init__(self, train=True):
        # mean & std CIFAR-10
        self.mean = (0.4914, 0.4822, 0.4465)
        self.std  = (0.2470, 0.2435, 0.2616)

        if train:
            self.aug = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.08, rotate_limit=15, p=0.6),
                # coarseDropout as requested
                A.CoarseDropout(max_holes=1, max_height=16, max_width=16,
                                min_holes=1, min_height=16, min_width=16,
                                fill_value=tuple(int(255*m) for m in self.mean),
                                p=0.5),
                A.Normalize(mean=self.mean, std=self.std),
                ToTensorV2(),
            ])
        else:
            self.aug = A.Compose([
                A.Normalize(mean=self.mean, std=self.std),
                ToTensorV2(),
            ])

    def __call__(self, img):
        # img is PIL Image: convert to np array
        if not isinstance(img, np.ndarray):
            img = np.array(img)
        out = self.aug(image=img)
        return out['image']

# ------------------------
# Layers: Depthwise Separable
# ------------------------
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, dilation=1, bias=False):
        super().__init__()
        # depthwise
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size=kernel_size, stride=stride,
                            padding=padding, dilation=dilation, groups=in_ch, bias=bias)
        # pointwise
        self.pw = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, padding=0, bias=bias)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = self.bn(x)
        return self.act(x)

# ------------------------
# The Model
# ------------------------
class CIFARNet(nn.Module):
    def __init__(self, num_classes=10, base_ch=24):
        """
        Design notes:
        - base_ch controls model width — keeps params low.
        - We use several dilated convs (dilations 1,2,4,8,16) then a final stride=2 conv (no MaxPool).
        - DepthwiseSeparableConv used for an early block.
        """
        super().__init__()
        C = base_ch  # small base channel count to stay under 200k params

        # C1: Standard conv 3x3
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, C, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )

        # C2: Depthwise Separable conv (one required layer)
        self.ds = DepthwiseSeparableConv(C, C*1 + 8, kernel_size=3, stride=1, padding=1)  # slight growth

        # C3: stack of dilated convs (one of these is the 'dilated conv' requirement)
        ch = C*1 + 8  # e.g., 32 if base_ch=24 -> 32
        self.dilated1 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=1, dilation=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )
        self.dilated2 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=2, dilation=2, bias=False),  # dilation=2
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )
        self.dilated3 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=4, dilation=4, bias=False),  # dilation=4
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )
        self.dilated4 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=8, dilation=8, bias=False),  # dilation=8
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )
        self.dilated5 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=16, dilation=16, bias=False),  # dilation=16
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )

        # C4: Final conv that downsamples using stride=2 (no MaxPool)
        # keep channel count moderate (no big explosion)
        self.down = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=2, padding=1, bias=False),  # this is the required stride-2 layer
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )

        # A couple more standard conv layers post-down to add capacity (no pooling)
        self.post1 = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True)
        )
        self.post2 = nn.Sequential(
            nn.Conv2d(ch, ch*2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(ch*2),
            nn.ReLU(inplace=True)
        )

        # GAP + FC
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch*2, num_classes)

        # initialize
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if getattr(m, "bias", None) is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv1(x)
        x = self.ds(x)
        x = self.dilated1(x)
        x = self.dilated2(x)
        x = self.dilated3(x)
        x = self.dilated4(x)
        x = self.dilated5(x)
        x = self.down(x)
        x = self.post1(x)
        x = self.post2(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# ------------------------
# Receptive field calculator (prints effective RF)
# ------------------------
def compute_receptive_field():
    # The model uses the following conv sequence (we will compute with stride & dilation)
    # We'll list the layers as (k,s,d) in order of application where k=kernel, s=stride, d=dilation
    seq = [
        (3,1,1),  # conv1
        (3,1,1),  # depthwise (dw) - effective kernel 3
        (1,1,1),  # pw 1x1 -> negligible RF increase but included for correctness
        (3,1,1),  # dilated1
        (3,1,2),  # dilated2
        (3,1,4),  # dilated3
        (3,1,8),  # dilated4
        (3,1,16), # dilated5
        (3,2,1),  # down stride=2
        (3,1,1),  # post1
        (3,1,1),  # post2
    ]
    rf = 1
    jump = 1
    for k,s,d in seq:
        rf = rf + (k-1)*d*jump
        jump = jump * s
    return rf

# ------------------------
# Training / evaluation
# ------------------------
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            loss = criterion(out, y)
            running_loss += loss.item() * x.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)
    return running_loss / total, 100.0 * correct / total

# ------------------------
# Main CLI
# ------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=180)
    parser.add_argument("--batch", type=int, default=128)
    parser.add_argument("--lr", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--base_ch", type=int, default=24, help="Base channel width (lower -> fewer params)")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--save", type=str, default="cifar_model.pth")
    args, _ = parser.parse_known_args()

    device = device_and_seed(args.seed)
    print("Device:", device)

    # Model & param count
    model = CIFARNet(num_classes=10, base_ch=args.base_ch).to(device)
    total_params = count_params(model)
    print("Model params:", total_params)
    assert total_params < 200_000, "Model exceeds 200k params! adjust base_ch."

    # RF print
    rf = compute_receptive_field()
    print("Estimated receptive field (pixels):", rf)
    if rf <= 44:
        print("WARNING: RF <= 44, adjust model to increase receptive field.")

    # Datasets (use torchvision dataset, but albumentations transforms)
    train_transform = AlbumentationsTransform(train=True)
    val_transform   = AlbumentationsTransform(train=False)

    # Datasets with lambda wrapper to apply albumentations
    class _CIFARWrapper(datasets.CIFAR10):
        def __init__(self, root, train, transform, download):
            super().__init__(root=root, train=train, transform=ToTensor(), download=download)
            self.alb_transform = transform
            # We'll ignore the inherited transform; use albementations on PIL images

        def __getitem__(self, index):
            img, target = self.data[index], int(self.targets[index])
            # img is numpy array in CIFAR dataset already (H,W,C uint8)
            img = img  # numpy array (H,W,C)
            img_t = self.alb_transform(img)
            return img_t, target

    root = "./data"
    train_set = _CIFARWrapper(root=root, train=True, transform=train_transform, download=True)
    val_set   = _CIFARWrapper(root=root, train=False, transform=val_transform, download=True)

    train_loader = DataLoader(train_set, batch_size=args.batch, shuffle=True, num_workers=args.workers, pin_memory=True)
    val_loader   = DataLoader(val_set, batch_size=args.batch, shuffle=False, num_workers=args.workers, pin_memory=True)

    # Criterion & optimizer & scheduler
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    best_acc = 0.0
    for epoch in range(1, args.epochs+1):
        st = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - st

        print(f"Epoch {epoch}/{args.epochs} | time {elapsed:.1f}s "
              f"| train loss {train_loss:.4f} acc {train_acc:.2f}% "
              f"| val loss {val_loss:.4f} acc {val_acc:.2f}%")

        # Save best
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "model_state": model.state_dict(),
                "epoch": epoch,
                "best_acc": best_acc,
                "args": vars(args)
            }, args.save)
            print(f"Saved best model (acc={best_acc:.2f}%) -> {args.save}")

    print("Training finished. Best val acc:", best_acc)

if __name__ == "__main__":
    main()


Device: cuda
Model params: 85914
Estimated receptive field (pixels): 77


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-96264895.py:61: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=1, max_height=16, max_width=16,
100%|██████████| 170M/170M [00:03<00:00, 42.8MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1/180 | time 21.3s | train loss 1.3544 acc 49.89% | val loss 1.2209 acc 57.34%
Saved best model (acc=57.34%) -> cifar_model.pth
Epoch 2/180 | time 21.1s | train loss 0.9733 acc 65.30% | val loss 1.1140 acc 62.98%
Saved best model (acc=62.98%) -> cifar_model.pth
Epoch 3/180 | time 20.4s | train loss 0.8634 acc 69.64% | val loss 0.9149 acc 68.53%
Saved best model (acc=68.53%) -> cifar_model.pth
Epoch 4/180 | time 19.8s | train loss 0.7925 acc 72.35% | val loss 0.8798 acc 70.81%
Saved best model (acc=70.81%) -> cifar_model.pth
Epoch 5/180 | time 19.8s | train loss 0.7538 acc 73.90% | val loss 1.0325 acc 67.49%
Epoch 6/180 | time 19.5s | train loss 0.7261 acc 74.84% | val loss 0.8896 acc 71.13%
Saved best model (acc=71.13%) -> cifar_model.pth
Epoch 7/180 | time 19.7s | train loss 0.7052 acc 75.82% | val loss 1.0018 acc 68.58%
Epoch 8/180 | time 19.2s | train loss 0.6866 acc 76.08% | val loss 0.9803 acc 69.76%
Epoch 9/180 | time 19.9s | train loss 0.6720 acc 76.90% | val loss 1.0577 a